### Задание на Четверку (Проведение исследований моделями семантической сегментации)

#### 4. Имплементация алгоритма машинного обучения

Простая CNN (U-Net light)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1),
            nn.ReLU()
        )

        self.pool = nn.MaxPool2d(2)

        self.enc2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU()
        )

        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)

        self.dec = nn.Sequential(
            nn.Conv2d(32, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 1, 1)
        )

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.pool(x1)

        x3 = self.enc2(x2)
        x4 = self.up(x3)

        out = self.dec(x4)
        return out

2. Простой Transformer (очень упрощённый ViT для сегментации)

In [ ]:
class SimpleViT(nn.Module):
    def __init__(self, img_size=256, patch_size=16, in_ch=3, dim=128):
        super().__init__()

        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.patch_embed = nn.Conv2d(in_ch, dim, patch_size, patch_size)

        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=dim, nhead=4),
            num_layers=4
        )

        self.head = nn.ConvTranspose2d(dim, 1, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.patch_embed(x) 

        b, c, h, w = x.shape
        x = x.flatten(2).permute(2, 0, 1) 

        x = self.transformer(x)

        x = x.permute(1, 2, 0).reshape(b, c, h, w)

        x = self.head(x)

        return x

Loss + device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loss_fn = nn.BCEWithLogitsLoss()

Train loop

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)
        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

Метрики

In [ ]:
def iou(pred, target):
    pred = (pred > 0.5).float()
    inter = (pred * target).sum()
    union = pred.sum() + target.sum() - inter
    return (inter + 1e-6) / (union + 1e-6)


def dice(pred, target):
    pred = (pred > 0.5).float()
    inter = (pred * target).sum()
    return (2 * inter + 1e-6) / (pred.sum() + target.sum() + 1e-6)


def acc(pred, target):
    pred = (pred > 0.5).float()
    return (pred == target).float().mean()

Evaluation

In [ ]:
def evaluate(model, loader):
    model.eval()

    iou_total = 0
    dice_total = 0
    acc_total = 0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            preds = torch.sigmoid(model(images))

            iou_total += iou(preds, masks).item()
            dice_total += dice(preds, masks).item()
            acc_total += acc(preds, masks).item()

    n = len(loader)

    return {
        "IoU": iou_total / n,
        "Dice": dice_total / n,
        "Acc": acc_total / n
    }

CNN

In [ ]:
model_cnn = SimpleCNN().to(device)
opt_cnn = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)

for epoch in range(10):
    loss = train_one_epoch(model_cnn, train_loader, opt_cnn)
    print(f"CNN Epoch {epoch+1}: {loss:.4f}")

Transformer

In [ ]:
model_vit = SimpleViT().to(device)
opt_vit = torch.optim.Adam(model_vit.parameters(), lr=1e-3)

for epoch in range(10):
    loss = train_one_epoch(model_vit, train_loader, opt_vit)
    print(f"ViT Epoch {epoch+1}: {loss:.4f}")

Оценка качества

In [ ]:
cnn_metrics = evaluate(model_cnn, val_loader)
vit_metrics = evaluate(model_vit, val_loader)

print("CNN:", cnn_metrics)
print("ViT:", vit_metrics)